In [1]:
!pip install google-genai

In [2]:
!pip install python-dotenv
!pip install pydantic 

In [6]:
from google import genai
import os
import json
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)
# print(api_key)

In [7]:
prompt = """
Generate 50 products for a toy store.

Return ONLY raw JSON.
Do NOT include markdown or ```.

Format:
[
  {
    "name": "string",
    "description": "string",
    "price": number,
    "quantity": number,
    "brand": "string",
    "category": "string"
  }
]

Rules:
- Categories should be like: "Educational Toys", "Action Figures", "Board Games", "Plush Toys", "Outdoor Toys"
- Price between 100 and 5000
- Quantity between 1 and 100
- Brand should be realistic (e.g., Lego, Hasbro, Fisher-Price)
- Make names creative and unique
"""

In [8]:
from pydantic import BaseModel, Field, ValidationError
from typing import List


class ProductSchema(BaseModel):
    name: str
    description: str
    price: float = Field(gt=0)
    quantity: int = Field(ge=0)
    brand: str
    category: str

In [9]:
response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=prompt,
    config={
        "temperature": 1.2
    }
)

In [10]:
import re
import json

raw_text = response.text

match = re.search(r"\[\s*{.*}\s*\]", raw_text, re.DOTALL)

if match:
    clean_text = match.group(0)
else:
    clean_text = raw_text

products_raw = json.loads(clean_text)

In [11]:
valid_products = []
invalid_products = []

for i, item in enumerate(products_raw):
    try:
        product = ProductSchema(**item)
        valid_products.append(product)
    except ValidationError as e:
        invalid_products.append({
            "index": i,
            "error": e.errors(),
            "data": item
        })

In [12]:
print(f"✅ Valid products: {len(valid_products)}")
print(f"❌ Invalid products: {len(invalid_products)}")

✅ Valid products: 49
❌ Invalid products: 0
